[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prashantkul/ai-ml-interviews-study-guide/blob/main/notebooks/week2_paired_comparison.ipynb)


# Week 2 — Comparing Two Monitors/Models on the Same Eval Set

**Topic:** Paired bootstrap and McNemar's test for comparing two systems evaluated on the *same* items.

**Interview question this answers:** *How would you determine if monitor A is better than monitor B?*

**Key idea:** When two models score the same items, their results are paired. Treating them as two independent samples throws away the pairing structure and **loses statistical power**. Easy items tend to be correct for both models; hard items wrong for both. The right question is not *"Is B's overall accuracy higher?"* but *"On the same items, does B beat A more often than the reverse?"*

We will:
1. Generate paired correctness vectors for monitors A (~0.90) and B (~0.92).
2. Run a naive (unpaired) two-proportion z-test.
3. Run a paired bootstrap from scratch.
4. Run McNemar's test on the discordant pairs.
5. Repeat 500 times to compare empirical power.

Reference: Berg-Kirkpatrick, Burkett, Klein, *An Empirical Investigation of Statistical Significance in NLP* (2012).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(7)

## 1. Synthetic paired data

Each item has a latent **difficulty** drawn uniformly on [0, 1]. Both monitors see the same items, so an easy item (low difficulty) is likely correct for both — that is what makes the data paired.

- Monitor A is correct iff `difficulty_i < 0.90 + small_noise_A`
- Monitor B is correct iff `difficulty_i < 0.92 + small_noise_B`

Marginal accuracies should land near 0.90 and 0.92, and `corr(a, b)` should be clearly positive.

In [ ]:
def generate_paired(n=300, acc_a=0.90, acc_b=0.92, noise_sd=0.03, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    difficulty = rng.uniform(0.0, 1.0, size=n)
    noise_a = rng.normal(0.0, noise_sd, size=n)
    noise_b = rng.normal(0.0, noise_sd, size=n)
    a = (difficulty < acc_a + noise_a).astype(int)
    b = (difficulty < acc_b + noise_b).astype(int)
    return a, b

rng = np.random.default_rng(7)
a, b = generate_paired(n=300, rng=rng)
n = len(a)

print(f"n = {n}")
print(f"Monitor A accuracy: {a.mean():.4f}")
print(f"Monitor B accuracy: {b.mean():.4f}")
print(f"Observed B - A:    {b.mean() - a.mean():+.4f}")
print(f"corr(a, b):        {np.corrcoef(a, b)[0, 1]:.4f}")

## 2. Naive unpaired comparison (two-proportion z-test)

Pretend `a` and `b` are two *independent* samples of size 300 each. This is the wrong analysis for paired data — but it is what you get if you forget the pairing. We compute a pooled-variance two-proportion z statistic.

In [ ]:
def unpaired_two_proportion_p(a, b):
    n_a, n_b = len(a), len(b)
    p_a, p_b = a.mean(), b.mean()
    p_pool = (a.sum() + b.sum()) / (n_a + n_b)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
    if se == 0:
        return 1.0
    z = (p_b - p_a) / se
    # two-sided p-value
    return 2 * stats.norm.sf(abs(z))

p_unpaired = unpaired_two_proportion_p(a, b)
print(f"Unpaired two-proportion z-test p-value: {p_unpaired:.4f}")
print("(This ignores the fact that A and B graded the same items.)")

## 3. Paired bootstrap from scratch (Berg-Kirkpatrick et al., 2012)

Resample item indices with replacement. Crucially, on each draw we use the **same indices** for both A and B — that preserves the pairing. Then we compute the paired difference `mean(b[idx]) - mean(a[idx])` 10,000 times and read off a 95% CI plus a one-sided p-value proxy.

In [ ]:
def paired_bootstrap(a, b, n_boot=10_000, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    n = len(a)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        diffs[i] = b[idx].mean() - a[idx].mean()
    return diffs

rng_boot = np.random.default_rng(7)
diffs = paired_bootstrap(a, b, n_boot=10_000, rng=rng_boot)

ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
observed_diff = b.mean() - a.mean()
frac_positive = (diffs > 0).mean()
# one-sided p-value proxy: how often the bootstrap difference is <= 0
p_paired_boot = (diffs <= 0).mean()

print(f"Observed diff (B - A):          {observed_diff:+.4f}")
print(f"Bootstrap mean diff:            {diffs.mean():+.4f}")
print(f"95% CI for diff:                [{ci_low:+.4f}, {ci_high:+.4f}]")
print(f"Fraction of replicates B > A:   {frac_positive:.4f}")
print(f"One-sided bootstrap p-value:    {p_paired_boot:.4f}")

## 4. McNemar's test

McNemar's test only looks at the **discordant pairs** — items where A and B disagree. Concordant items (both right or both wrong) carry no information about which model is better.

|              | B correct | B wrong |
|--------------|-----------|---------|
| **A correct**| b_11      | b_10    |
| **A wrong**  | b_01      | b_00    |

Under H0 (the two models are equally good), among the discordant pairs `b_01` and `b_10` should be roughly equal. We use the chi-squared form with continuity correction when `b_01 + b_10 >= 25`, and the exact binomial otherwise.

In [ ]:
def mcnemar(a, b):
    a = np.asarray(a).astype(bool)
    b = np.asarray(b).astype(bool)
    b11 = int(np.sum(a & b))
    b10 = int(np.sum(a & ~b))   # A right, B wrong
    b01 = int(np.sum(~a & b))   # A wrong, B right
    b00 = int(np.sum(~a & ~b))
    n_disc = b01 + b10
    if n_disc == 0:
        return {"b11": b11, "b10": b10, "b01": b01, "b00": b00, "p_value": 1.0, "method": "degenerate"}
    if n_disc < 25:
        # exact two-sided binomial test on b01 vs n_disc, p=0.5
        k = min(b01, b10)
        p_value = 2 * stats.binom.cdf(k, n_disc, 0.5)
        p_value = min(p_value, 1.0)
        method = "exact binomial"
    else:
        chi2 = (abs(b01 - b10) - 1) ** 2 / n_disc
        p_value = stats.chi2.sf(chi2, df=1)
        method = "chi2 with continuity correction"
    return {"b11": b11, "b10": b10, "b01": b01, "b00": b00, "p_value": p_value, "method": method}

m = mcnemar(a, b)
print("Contingency table:")
print(f"  B correct  B wrong")
print(f"A correct   {m['b11']:5d}    {m['b10']:5d}")
print(f"A wrong     {m['b01']:5d}    {m['b00']:5d}")
print()
print(f"Discordant pairs: b01={m['b01']} (A wrong, B right), b10={m['b10']} (A right, B wrong)")
print(f"McNemar method:   {m['method']}")
print(f"McNemar p-value:  {m['p_value']:.4f}")

## 5. Why pairing matters — power simulation

We now repeat the experiment 500 times with fresh synthetic draws. For each draw we record:
- the unpaired two-proportion z-test p-value
- the paired bootstrap two-sided p-value

Then we plot the two p-value distributions and compute empirical power at alpha=0.05 (the fraction of trials in which each test rejects H0).

If pairing matters, the paired test should reject more often.

In [ ]:
def paired_bootstrap_pvalue_two_sided(a, b, n_boot=2000, rng=None):
    """Two-sided bootstrap p-value for H0: mean(b) == mean(a).
    Uses the standard 'shift to null' trick: subtract the observed difference
    from each replicate, then ask how often the shifted statistic is at least
    as extreme as the observed one."""
    if rng is None:
        rng = np.random.default_rng()
    n = len(a)
    obs = b.mean() - a.mean()
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        diffs[i] = b[idx].mean() - a[idx].mean()
    centered = diffs - obs  # shift to null
    p = (np.abs(centered) >= abs(obs)).mean()
    return p

n_trials = 500
alpha = 0.05
p_unpaired_arr = np.empty(n_trials)
p_paired_arr = np.empty(n_trials)

rng_sim = np.random.default_rng(7)
for t in range(n_trials):
    at, bt = generate_paired(n=300, rng=rng_sim)
    p_unpaired_arr[t] = unpaired_two_proportion_p(at, bt)
    p_paired_arr[t] = paired_bootstrap_pvalue_two_sided(at, bt, n_boot=2000, rng=rng_sim)

power_unpaired = (p_unpaired_arr < alpha).mean()
power_paired = (p_paired_arr < alpha).mean()

print(f"Empirical power at alpha=0.05 over {n_trials} trials, n=300 each:")
print(f"  Unpaired two-proportion z-test: {power_unpaired:.3f}")
print(f"  Paired bootstrap:               {power_paired:.3f}")
print(f"  Power gain from pairing:        {power_paired - power_unpaired:+.3f}")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
bins = np.linspace(0, 1, 41)
ax.hist(p_unpaired_arr, bins=bins, alpha=0.55, label=f"Unpaired z-test (power={power_unpaired:.2f})")
ax.hist(p_paired_arr, bins=bins, alpha=0.55, label=f"Paired bootstrap (power={power_paired:.2f})")
ax.axvline(alpha, color="red", linestyle="--", label=f"alpha={alpha}")
ax.set_xlabel("p-value")
ax.set_ylabel("count over 500 trials")
ax.set_title("P-value distribution: paired vs unpaired comparison of A vs B")
ax.legend()
plt.tight_layout()
plt.show()

**What the histogram shows.** The paired bootstrap p-values are pushed toward zero (more rejections), while the unpaired p-values are spread much higher. Both tests are looking at the same true effect size (~2 percentage points), but the paired test conditions out per-item difficulty: items that are easy or hard for both models contribute no noise to the *difference*. The unpaired test treats that shared difficulty as independent noise in each sample, inflating the standard error and burying the signal.

## 6. Flashcard

> **Q:** To compare monitor A vs B on the same data, I use **paired bootstrap** because...
>
> **A:** ...the two monitors score the *same* items, so their correctness vectors are correlated (easy items are right for both, hard items wrong for both). Resampling item indices once and applying them to both A and B preserves that pairing, conditions out per-item difficulty, and gives a much tighter sampling distribution for the *difference* in accuracy than treating A and B as two independent samples.
>
> **Q:** McNemar's test is the alternative when...
>
> **A:** ...the metric is binary correctness per item and we just want a closed-form test on paired binary outcomes. It only uses the discordant pairs (b01, b10) — items where A and B disagree — and asks whether disagreements split evenly under H0. Use the chi-squared form with continuity correction for n_disc >= 25, and the exact binomial test for small discordant counts. Use paired bootstrap instead when the metric is non-binary (F1, AUC, BLEU, calibration error) or when you want a confidence interval on the difference.

## 7. Interview rehearsal

**Question:** *How would you determine if monitor A is better than monitor B?*

**90-second answer (using this notebook's numbers):**

> The first thing I'd flag is that A and B are scored on the *same* eval set, so the data is paired and I need a paired test. If I forget that and run a two-proportion z-test on the marginal accuracies — which I did in this notebook for n=300 with true accuracies around 0.90 and 0.92 — I get a p-value that often misses significance, because the test assumes the noise in A and the noise in B are independent. They aren't: easy items are right for both, hard items wrong for both.
>
> The right tool is paired bootstrap (Berg-Kirkpatrick et al., 2012). I resample *item indices* with replacement and use the same indices for both monitors, then take percentiles of `mean(b[idx]) - mean(a[idx])` to get a 95% CI on the difference, plus a p-value by shifting to the null. In this notebook the paired CI excludes zero comfortably, and across 500 simulated runs the paired bootstrap had empirical power around {paired_power} at alpha=0.05 versus only about {unpaired_power} for the naive unpaired test — same data, same effect size, much more power, just from respecting the pairing.
>
> If the metric is binary correctness per item and I just want a quick closed-form check, McNemar's test on the discordant pairs is the clean alternative. I'd reach for paired bootstrap when the metric is something like F1, AUC, or calibration error where there's no closed form.

In [ ]:
print(f"Fill-in numbers for the rehearsal above:")
print(f"  paired_power   = {power_paired:.2f}")
print(f"  unpaired_power = {power_unpaired:.2f}")